In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Working Environment

In [ ]:
!pip install langchain==1.0.5 -q
!pip install -U langchain langchain-community -q 
!pip install -U langchain-google-genai -q

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
from langchain_google_genai import ChatGoogleGenerativeAI 

In [ ]:
llm_model = "gemini-2.5-flash"

In [ ]:
user_secrets = UserSecretsClient()
os.environ["GOOGLE_API_KEY"] = user_secrets.get_secret("gimini_key_2")
chat = ChatGoogleGenerativeAI(
    model=llm_model,
    temperature=0
)

In [ ]:
import pandas as pd 

In [ ]:
data_path =  '/kaggle/input/datasets/abdullahelsaeed/product-review-csv/product review.csv'
df = pd.read_csv(data_path)
df.head()

# LLM Chain

In [ ]:
!pip install langchain-classic -q

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains import LLMChain

In [ ]:
prompt = ChatPromptTemplate.from_template(
    "What is the best name to describe a company that makes {product}? \
    Return only the best name without any explanation."
)

In [ ]:
print(prompt)

In [ ]:
chain = LLMChain(llm=chat,prompt=prompt)

In [ ]:
product = "Deep Learning GPUs"
chain.run(product)

## There is anther way

In [ ]:
chain = prompt | chat

In [ ]:
product = "Deep Learning GPUs"
result = chain.invoke({"product": product})

In [ ]:
print(result.content)

# Sequential Chains 


## 1.Simple Sequential Chain 

In [ ]:
# prompt template 1
first_prompt = ChatPromptTemplate.from_template(
    "What is the best name to describe a company that makes {product}? \
    Return only the best name without any explanation."
)
# Chain 1 
chain_one = first_prompt | chat

In [ ]:
# prompt template 2
second_prompt = ChatPromptTemplate.from_template(
    "Write a 30 words description for the following \
    company:{company_name}"
)
# chain 2
chain_two = second_prompt | chat

In [ ]:
overall_simple_chain = chain_one | chain_two

In [ ]:
result = overall_simple_chain.invoke({"product": product})

In [ ]:
result.content

## A more stable method

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

parser = StrOutputParser()

chain_one = first_prompt | chat | parser

overall_simple_chain = (
    chain_one
    | RunnableLambda(lambda name: {"company_name": name})
    | second_prompt
    | chat
    | parser
)

result = overall_simple_chain.invoke({"product": product})
print(result)

# 2.Complex Sequential Chain

In [ ]:
# prompt template 1: translate to english
first_prompt = ChatPromptTemplate.from_template(
    "Translate the following review to English:"
    "\n\n{Review}"
)

In [ ]:
# chain 1: input= Review and output= English_Review

translate_chain = first_prompt | chat | StrOutputParser()

chain_one = {
    "Arabic_Review": translate_chain
}

In [ ]:
second_prompt = ChatPromptTemplate.from_template(
    "Can you summarize the following review in 1 sentence:"
    "\n\n{English_Review}"
)

In [ ]:
# chain 2: input= English_Review and output= summary
summarize_chain = second_prompt | chat | StrOutputParser()

chain_two = {
    "summary" : summarize_chain
}

In [ ]:
# prompt template 3: translate to english
third_prompt = ChatPromptTemplate.from_template(
    "What language is the following review:\n\n{Review}"
)

In [ ]:
language_chain = third_prompt | chat | StrOutputParser()

chain_three = {
    "language": language_chain
}

In [ ]:
# prompt template 4: follow up message
fourth_prompt = ChatPromptTemplate.from_template(
    "Write a follow up response to the following "
    "summary in the specified language:"
    "\n\nSummary: {summary}\n\nLanguage: {language}"
)

In [ ]:
# chain 4: input= summary, language and output= followup_message
followup_message_chain = fourth_prompt | chat | StrOutputParser()

chain_four = {
    "followup_message" : followup_message_chain
}

In [ ]:
from langchain_core.runnables import RunnablePassthrough

In [ ]:
overall_chain = (
    RunnablePassthrough()
    .assign(English_Review=translate_chain)
    .assign(summary=summarize_chain)
    .assign(language=language_chain)
    .assign(followup_message=followup_message_chain)
)

In [ ]:
review = "J'ai vraiment apprécié ce film. L'histoire était captivante du début à la fin et les acteurs ont livré des performances très convaincantes. La photographie était magnifique et la musique renforçait parfaitement l'ambiance. Cependant, la fin m'a semblé un peu rapide. Malgré cela, je recommande fortement ce film."

In [ ]:
overall_chain.invoke({"Review":review})

- use "Return ONLY the translated text with no explanation" to make LLM return the result only 

# 4. Router Chain

### [Runnables in Detail](https://medium.com/@mrcoffeeai/runnables-in-detail-0c5d417bc2a1)

In [ ]:
physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise\
and easy to understand manner. \
When you don't know the answer to a question you admit\
that you don't know.

Here is a question:
{input}"""


math_template = """You are a very good mathematician. \
You are great at answering math questions. \
You are so good because you are able to break down \
hard problems into their component parts, 
answer the component parts, and then put them together\
to answer the broader question.

Here is a question:
{input}"""

history_template = """You are a very good historian. \
You have an excellent knowledge of and understanding of people,\
events and contexts from a range of historical periods. \
You have the ability to think, reflect, debate, discuss and \
evaluate the past. You have a respect for historical evidence\
and the ability to make use of it to support your explanations \
and judgements.

Here is a question:
{input}"""


computerscience_template = """ You are a successful computer scientist.\
You have a passion for creativity, collaboration,\
forward-thinking, confidence, strong problem-solving capabilities,\
understanding of theories and algorithms, and excellent communication \
skills. You are great at answering coding questions. \
You are so good because you know how to solve a problem by \
describing the solution in imperative steps \
that a machine can easily interpret and you know how to \
choose a solution that has a good balance between \
time complexity and space complexity. 

Here is a question:
{input}"""

In [ ]:
prompt_infos = [
    {
        "name": "physics", 
        "description": "Good for answering questions about physics", 
        "prompt_template": physics_template
    },
    {
        "name": "math", 
        "description": "Good for answering math questions", 
        "prompt_template": math_template
    },
    {
        "name": "History", 
        "description": "Good for answering history questions", 
        "prompt_template": history_template
    },
    {
        "name": "computer science", 
        "description": "Good for answering computer science questions", 
        "prompt_template": computerscience_template
    }
]

In [ ]:
from langchain_core.runnables import RunnablePassthrough, RunnableBranch

In [ ]:
destination_chains = {}

for p_info in prompt_infos:
    name = p_info["name"]
    prompt_template = p_info["prompt_template"]

    prompt = ChatPromptTemplate.from_template(prompt_template)

    chain = prompt | chat | StrOutputParser()

    destination_chains[name] = chain

In [ ]:
destination_chains

In [ ]:
destinations = [f"{p['name']}: {p['description']}" for p in prompt_infos]
destinations_str = "\n".join(destinations)

In [ ]:
destinations_str

In [ ]:
default_prompt = ChatPromptTemplate.from_template("{input}")

default_chain =  

In [ ]:
default_prompt = ChatPromptTemplate.from_template("{input}")

default_chain = default_prompt | chat | StrOutputParser()

In [ ]:
MULTI_PROMPT_ROUTER_TEMPLATE = """Given a raw text input to a \
language model select the model prompt best suited for the input. \
You will be given the names of the available prompts and a \
description of what the prompt is best suited for. \
You may also revise the original input if you think that revising\
it will ultimately lead to a better response from the language model.

<< FORMATTING >>
Return a markdown code snippet with a JSON object formatted to look like:
```json
{{{{
    "destination": string \ name of the prompt to use or "DEFAULT"
    "next_inputs": string \ a potentially modified version of the original input
}}}}
```

REMEMBER: "destination" MUST be one of the candidate prompt \
names specified below OR it can be "DEFAULT" if the input is not\
well suited for any of the candidate prompts.
REMEMBER: "next_inputs" can just be the original input \
if you don't think any modifications are needed.

<< CANDIDATE PROMPTS >>
{destinations}

<< INPUT >>
{{input}}

<< OUTPUT (remember to include the ```json)>>"""

In [ ]:
router_prompt = ChatPromptTemplate.from_template(
     MULTI_PROMPT_ROUTER_TEMPLATE
)


In [ ]:
router_chain = router_prompt | chat | StrOutputParser()

In [ ]:
from langchain_core.runnables import RunnableLambda
overall_router = (
    RunnablePassthrough()
    .assign(
        route=(
            router_chain
            |  RunnableLambda(lambda s: s.strip().lower())
        )
    )
    | RunnableBranch(
        (lambda x: x["route"] == "physics", destination_chains["physics"]),
        (lambda x: x["route"] == "math", destination_chains["math"]),
        (lambda x: x["route"] == "history", destination_chains["History"]),
        (lambda x: x["route"] in ["computer science", "computer_science"], destination_chains["computer science"]),
        default_chain
    )
)

In [ ]:
overall_router.invoke({
    "input": "What is 9 * 7?",
    "destinations": destinations_str
})